# 04 — Sensitivity, specification curves, and what would change the answer

A single estimate from either design is a choice, not a result. This notebook varies every
discretionary decision and reports the resulting spread. Where the spread is narrow the design is
robust; where it is wide, the number should be reported as a range or not at all.

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
FIGS = REPO / "report" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 25)

from cid import io as cidio, viz
from IPython.display import display, Image


def show(name, width=780):
    """Display a figure the analysis just wrote, so output is deterministic across backends."""
    path = FIGS / name
    if path.exists():
        display(Image(filename=str(path), embed=True, width=width))
    else:
        print("missing figure:", path)

In [2]:
from cid.sc import weights
from cid.did import cs, twfe

mpdta = cidio.load_mpdta()
smoking = cidio.load_smoking()
PRED = ["lnincome", "beer", "age15to24", "lnprice"]

## 1. Minimum wage: how much does each discretionary choice move the estimate?

Two axes: the comparison-group rule and the reweighting estimator. Each is a decision a practitioner
makes and often does not disclose.

In [3]:
run = {}
for control in ["not-yet-treated", "never-treated"]:
    for est in ["simple", "reg", "ipw", "dr"]:
        surf = cs.att_surface(mpdta, covars=["lpop"], estimator=est, control=control, n_boot=150)
        run[(control, est)] = cs.aggregate(surf, "simple").att.iloc[0]
tab = pd.Series(run).rename_axis(["control_rule", "estimator"]).to_frame("overall_ATT").reset_index()
tab["as_%_change"] = (100 * (np.exp(tab.overall_ATT) - 1)).round(2)
tab

,control_rule,estimator,overall_ATT,as_%_change
0,not-yet-treated,simple,-0.039764,-3.90
1,not-yet-treated,reg,-0.041375,-4.05
2,not-yet-treated,ipw,-0.042674,-4.18
3,not-yet-treated,dr,-0.041305,-4.05
4,never-treated,simple,-0.039951,-3.92
5,never-treated,reg,-0.041969,-4.11
6,never-treated,ipw,-0.045153,-4.41
7,never-treated,dr,-0.041614,-4.08


In [4]:
print("range across %d specifications: %+.4f to %+.4f (spread %.4f)"
      % (len(tab), tab.overall_ATT.min(), tab.overall_ATT.max(), tab.overall_ATT.max() - tab.overall_ATT.min()))
for rule, blk in tab.groupby("control_rule"):
    print("  %-16s estimator spread %.4f" % (rule, blk.overall_ATT.max() - blk.overall_ATT.min()))
for est in ["simple", "reg", "ipw", "dr"]:
    blk = tab[tab.estimator == est]
    print("  %-7s control-rule gap  %.4f" % (est, abs(blk.overall_ATT.max() - blk.overall_ATT.min())))
print("naive static TWFE for comparison: %+.4f" % twfe.static_twfe(mpdta)["estimate"])

range across 8 specifications: -0.0452 to -0.0398 (spread 0.0054)
  never-treated    estimator spread 0.0052
  not-yet-treated  estimator spread 0.0029
  simple  control-rule gap  0.0002
  reg     control-rule gap  0.0006
  ipw     control-rule gap  0.0025
  dr      control-rule gap  0.0003


naive static TWFE for comparison: -0.0365


The spread across all eight specifications is 0.0054 log points — half a percent of teen
employment, and smaller than the standard error of any single `(g, t)` cell. Neither axis dominates:
holding the control rule fixed, the four reweighting strategies move the answer by 0.003-0.005;
holding the estimator fixed, switching control rule moves it by 0.0002-0.0025. With one covariate and
309 never-treated counties, the propensity and outcome models have almost nothing to disagree about.
That is a property of this panel, not a general guarantee.

What deserves the attention is the last line. The naive static TWFE (-0.0365) is *attenuated*
relative to the group-time estimates (-0.0413), and the direction matches notebook 02 section 3:
contaminated pairwise comparisons average +0.003 against -0.012 for clean ones, so they drag the
pooled number toward zero. Here the bias is about 12% of the estimate — survivable. In a panel with a
steeper effect path, or with early-treated units large enough to dominate the weights, the same
mechanism is the difference between a significant result and a null.

## 2. Synthetic control: leave-one-donor-out stability

If the estimate depends on whether a particular state is allowed to be a donor, the counterfactual
is fragile.

In [5]:
donors = [d for d in smoking.state.unique() if d != "California"]
ref = weights.trajectory_fit(smoking)
loo = []
for d in donors:
    sub = smoking[smoking.state != d]
    try:
        r = weights.trajectory_fit(sub)
    except (RuntimeError, ValueError):
        continue
    loo.append({"dropped_donor": d, "effect_2000": r["final_year_effect"],
                "pre_rmspe": r["pre_rmspe"], "mean_effect": r["mean_annual_effect"],
                "weight_it_had": float(ref["weights"].get(d, 0.0))})
loo = pd.DataFrame(loo).sort_values("effect_2000")
loo.round(3)

,dropped_donor,effect_2000,pre_rmspe,mean_effect,weight_it_had
3,Connecticut,-27.933,1.834,-20.265,0.079
2,Colorado,-27.149,1.700,-19.946,0.030
21,New Mexico,-26.917,1.702,-19.762,0.000
11,Kentucky,-26.917,1.709,-19.760,0.000
23,North Dakota,-26.908,1.699,-19.757,0.000
34,Virginia,-26.906,1.699,-19.754,0.000
10,Kansas,-26.906,1.698,-19.754,0.000
35,West Virginia,-26.906,1.698,-19.753,0.000
18,Nebraska,-26.905,1.699,-19.752,0.000
6,Idaho,-26.905,1.700,-19.751,0.000


In [6]:
full = ref["final_year_effect"]
print("full-pool estimate        : %.2f" % full)
print("leave-one-out min / max   : %.2f / %.2f" % (loo.effect_2000.min(), loo.effect_2000.max()))
print("runs within 0.05 packs    : %d of %d" % (int(((loo.effect_2000 - full).abs() < 0.05).sum()), len(loo)))
print("moved by more than 1 pack : %s" % ", ".join(loo[(loo.effect_2000 - full).abs() > 1].dropped_donor))
print("worst pre-fit degradation : %.2f -> %.2f" % (loo.pre_rmspe.min(), loo.pre_rmspe.max()))

full-pool estimate        : -26.90
leave-one-out min / max   : -27.93 / -24.85
runs within 0.05 packs    : 32 of 38
moved by more than 1 pack : Connecticut, Utah, Montana
worst pre-fit degradation : 1.70 -> 2.43


This test is much weaker evidence than it first appears, and the honest reading is in the third
printed line: most deletions leave the estimate unchanged. That is not stability — it is that most
donors were never used, so removing them cannot matter. The informative rows are the six states that
actually carried weight, and there the sensitivity is real. Dropping Montana moves the 2000 gap from
-26.90 to -24.85 and the mean annual effect from -19.75 to -17.90; dropping Utah gives -25.55, with
the pre-period fit degrading from 1.70 to 2.43 — the tell that Utah was doing matching work rather
than sitting idle. Dropping Nevada, the third-largest weight, barely moves it (-26.78): the remaining
donors substitute for it, which is the useful kind of stability.

So the correct summary is not "the result is robust to donor-pool perturbation". It is: *the result
depends on about half a dozen states, and removing the largest of them moves it by roughly 8%.* That
is a materially different claim, and it is the one a reviewer should be given.

## 3. Specification curve across both designs

One table, every defensible choice, sorted by the estimate. The shape of this table is the result
— a single number taken from the middle of it is not.

In [7]:
traj = weights.trajectory_fit(smoking)
rows = [{"design": "synthetic_control", "spec": "trajectory 1970-88",
         "estimate": traj["mean_annual_effect"], "unit": "packs/capita/yr",
         "pre_fit": traj["pre_rmspe"]}]
for fs in (1970, 1980, 1985):
    r = weights.fit(smoking, predictors=PRED, feature_start=fs)
    rows.append({"design": "synthetic_control", "spec": f"features {fs}-88",
                 "estimate": r["mean_annual_effect"], "unit": "packs/capita/yr", "pre_fit": r["pre_rmspe"]})
for row in tab[tab.estimator == "dr"].itertuples():
    rows.append({"design": "did_mpdt", "spec": f"C&S dr / {row.control_rule}",
                 "estimate": row.overall_ATT, "unit": "log points", "pre_fit": np.nan})
rows.append({"design": "did_mpdt", "spec": "naive static TWFE",
             "estimate": twfe.static_twfe(mpdta)["estimate"], "unit": "log points", "pre_fit": np.nan})

curve = pd.DataFrame(rows).sort_values("estimate")
curve

,design,spec,estimate,unit,pre_fit
3,synthetic_control,features 1985-88,-24.646952,packs/capita/yr,7.663686
1,synthetic_control,features 1970-88,-24.484623,packs/capita/yr,8.754203
2,synthetic_control,features 1980-88,-21.193456,packs/capita/yr,5.995491
0,synthetic_control,trajectory 1970-88,-19.750874,packs/capita/yr,1.697697
5,did_mpdt,C&S dr / never-treated,-0.041614,log points,NaN
4,did_mpdt,C&S dr / not-yet-treated,-0.041305,log points,NaN
6,did_mpdt,naive static TWFE,-0.036549,log points,NaN


In [8]:
for design, blk in curve.groupby("design"):
    print(f"{design}: {blk.estimate.min():+.3f} to {blk.estimate.max():+.3f} "
          f"(range {blk.estimate.max() - blk.estimate.min():.3f}), n_specs={len(blk)}")

did_mpdt: -0.042 to -0.037 (range 0.005), n_specs=3
synthetic_control: -24.647 to -19.751 (range 4.896), n_specs=4


Two designs, two very different pictures. The minimum wage estimate is stable to within 0.005
log points across every defensible choice — the design is insensitive to the analyst. The synthetic
control estimate ranges from -19.8 to -24.6 packs, a spread of 4.9 packs that is 25% of the estimate
itself, and the ordering is close to what pre-period fit predicts: the specification with the best
pre-fit (1.70) gives the smallest effect and every worse-fitting one gives a larger effect, though
the three feature windows are not perfectly monotone among themselves. The direction of the pattern
is the point: specifications that match California badly overstate the tax's effect, because their
residual mismatch is counted as treatment effect.

That is the generalisable finding of this repository, and it is the one to put in a memo: *in the
absence of a randomised trial, the width of the specification curve — not the point estimate — is the
honest measure of what the data supports.*

## 4. The assumptions that cannot be tested

Everything above varies *method*. These vary *the world*, and no reweighting fixes them.

| Threat | Design A | Design B |
|---|---|---|
| Differential selection | Counties that raised the wage may differ in unobserved ways that correlate with employment trends. Partially addressed by the pre-trend test, not resolved. | California's 1988 decision may reflect an underlying anti-smoking culture that no donor matches. The in-time placebo is the closest thing to evidence on this. |
| Spillovers | Teen employment can move across county lines within a metro. Neighbouring-county correlation would need a spatial design to rule out. | Cross-border purchasing and mail order shift sales between states, which biases *toward* finding an effect. |
| Simultaneous policy | Other labour-market policies changing in 2004-07. | Federal excise changes and 1990s tobacco litigation, partly absorbed by the `retprice`/`lnprice` feature. |
| Interference in controls | Not-yet-treated counties anticipate their own increase. | Donor states respond to California's tax by adjusting their own. |

The pre-trend test in notebook 02 passed: the joint Wald over the two pre-treatment coefficients the
binned specification can actually identify gives p = 0.31, and none of the five
Callaway-Sant'Anna pre-treatment placebo cells has a 95% interval excluding zero (largest |t| = 1.6).
That is a failure to refute, not a confirmation. The honest statement is "no detectable divergence in
the pre-treatment window at this sample size", and the power behind it is bounded by 191 treated
counties observed over five periods, of which at most four precede any given cohort's treatment.

## 5. Headline numbers, reported the way I would put them in a memo

- **Minimum wage and teen employment.** ATT across all post periods: about **-4%** (95% CI roughly
  [-0.06, -0.02] on the doubly robust estimate), with every covariate-adjustment strategy and both
  control rules inside a 0.005 log-point band. The effect builds over about two years rather than
  appearing at once. Naive static two-way FE lands at -0.0365, 12% attenuated, because contaminated
  pairwise comparisons pull it toward zero.
- **Prop 99 and California cigarette sales.** Roughly **-20 packs per capita per year**, closing at
  **-26.9** by 2000, against a synthetic counterfactual that matches pre-1989 to within 1.5% error.
  Direction and order of magnitude hold across four specifications. The formal inference is weak:
  permutation p = 0.05 against all donors but 0.17 against the twelve that fit at least as well, and
  in-time placebos (sample truncated at 1988) manufacture -4.3 packs a year from nothing. Report the
  sign and the magnitude, not a p-value.

## 6. What I would do with more time

1. Cluster-robust block bootstrap over *whole unit trajectories* for the Callaway-Sant'Anna
   aggregations, so the overall CI respects that counties contribute to several `(g, t)` cells.
2. Covariate-adjusted synthetic control, or a matrix-completion counterfactual, so the estimate does
   not hinge on a half-dozen donors.
3. A power calculation for the pre-trend test: how large a violation would this design actually
   detect? Without it "we failed to reject" is unquantified.
4. Spatially defined control sets — neighbouring counties and adjacent states excluded — to probe the
   spillover assumption directly.